# Member 3: Employee Demand Prediction - Analysis & Training

This notebook contains the exploratory data analysis and model training for the Employee Demand Prediction service.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Set plot style
plt.style.use('ggplot')
sns.set_palette("husl")

## 1. Load Data

In [ ]:
data_path = '../data/employee_demand_dataset.csv'
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)
Visualizing patterns to understand what drives staffing needs.

In [ ]:
# 2.1 Correlation Heatmap
plt.figure(figsize=(12, 8))
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# 2.2 Fuel Demand vs Employee Count (Colored by Weekend/Holiday)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='predicted_fuel_demand', y='employee_count', hue='is_weekend', ax=ax1, alpha=0.6)
ax1.set_title("Fuel Demand vs Staffing (Weekend vs Weekday)")

sns.scatterplot(data=df, x='predicted_fuel_demand', y='employee_count', hue='is_holiday', ax=ax2, alpha=0.6)
ax2.set_title("Fuel Demand vs Staffing (Holiday vs Normal)")

plt.show()

In [ ]:
# 2.3 Average Staffing by Month and Weather
plt.figure(figsize=(14, 6))
sns.lineplot(data=df, x='month', y='employee_count', hue='weather', marker='o')
plt.title("Seasonality: Staffing Trends by Month and Weather")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.show()

## 3. Model Training

In [ ]:
feature_cols = [
    'month', 'day_of_week', 'day_of_month', 'week_of_year',
    'is_weekend', 'is_month_end', 'is_holiday', 'is_vacation',
    'is_day_before_holiday', 'is_friday',
    'weather', 'temperature', 'predicted_fuel_demand'
]
target_col = 'employee_count'

X = df[feature_cols]
y = df[target_col]

categorical_features = ['weather']
numeric_features = [col for col in feature_cols if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=150, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
print("Model trained successfully!")

## 4. Evaluation & Results Visualization

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.4f} employees")
print(f"R2 Score: {r2:.4f}")

# 4.1 Actual vs Predicted Plot
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Staff Count")
plt.ylabel("Predicted Staff Count")
plt.title("Actual vs Predicted Staffing Requirements")
plt.show()

In [ ]:
# 4.2 Residual Distribution
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True, color='crimson')
plt.title("Distribution of Prediction Errors (Residuals)")
plt.xlabel("Error (Actual - Predicted)")
plt.show()